# DiffuQR - end-to-end testing (reference-free)

Runs the full deployable pipeline on random real buggy frames, with **no reference photo and no
ground-truth geometry** anywhere in the loop:

```
frame -> SAM3 mask ("QR Code") -> quad fit -> version + orientation + corner refinement
      -> contrast stretch -> module-space input -> diffusion (checkpoint from the Hub) -> RS decode
```

Ground truth (payload, module grid) is used **only to score** the outputs. Library code: `diffuqr/pipeline.py`
(geometry), `diffuqr/diffusion.py` (model, sampling, decoding).

Compared on the same frames:

| column | what it is |
|---|---|
| `opencv`, `pyzbar` | off-the-shelf decoders on the full native frame |
| `reader` | learning-free module reader + soft RS, on OUR geometry (mask pipeline) |
| `model_stretch` | diffusion on our geometry, contrast-stretched input (the deployable system) |
| `model_raw` | same, no contrast stretch (ablation) |
| `oracle` | diffusion on the reference-aligned input from `modspace/real_*.npz` (upper bound; only frames that aligned) |

**Before running on Kaggle**
1. Settings: Accelerator **GPU T4** (or P100), **Internet on**.
2. Add-ons -> Secrets: `HF_TOKEN` (read on the dataset + model repos; must also have access to the gated
   `facebook/sam3`, or add a separate `SAM3_HF_TOKEN` secret).
3. Run all. About 15-25 min for 2 x 100 frames (SAM3 on GPU, geometry on CPU in parallel, model on GPU).

Nothing is pushed anywhere; results are written to `/kaggle/working/testing/` (the CSV has no payload).

In [ ]:
# ---- Configuration ----------------------------------------------------------
GIT_REF = "main"                       # branch / tag / commit of github.com/Rakshit-Sawarn-iitb/DiffuQR
DATASET_REPO = "RakshitAi/IITB-BuggyQR"           # private; read-only here
MODEL_REPO = "RakshitAi/IITB-BuggyQR-diffusion"   # checkpoints
CKPT = "modspace_diff_v1/last.pt"

SESSIONS = ("buggy01_20260917_01", "buggy02_20261018_01")
N_PER_SESSION = 100        # random frames per session (all saved frames are candidates, with or without a box)
SEED = 0
K_LIST = (1,)              # diffusion samples per frame (verifier decode); e.g. (1, 4) costs 4x
SAMPLE_STEPS = 20          # DDIM steps
MASK_SOURCE = "sam3"       # "sam3" (deployable) | "box" (ablation: filled SAM3 bounding box as the mask)
SAM3_THRESHOLD = 0.5
SAM3_MASK_THRESHOLD = 0.5
GEOM_TOP = 3               # versions refined finely after the coarse stage (diffuqr.pipeline.locate)
SMOKE = False              # True: 4 frames per session, 4 DDIM steps (quick end-to-end check)

if SMOKE:
    N_PER_SESSION, SAMPLE_STEPS = 4, 4
print(f"MASK_SOURCE={MASK_SOURCE}  N_PER_SESSION={N_PER_SESSION}  K={K_LIST}  steps={SAMPLE_STEPS}  SMOKE={SMOKE}")

In [ ]:
# ---- Environment: install the diffuqr library, paths, tokens ------------------
import os, sys, subprocess
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
if ON_KAGGLE:
    if "diffuqr" in sys.modules:
        raise RuntimeError("An older diffuqr is already imported in this kernel. "
                           "Run -> Restart & clear outputs, then run all cells again.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segno", "reedsolo", "diffusers", "pyzbar"], check=True)
    subprocess.run("apt-get install -y -qq libzbar0 > /dev/null 2>&1", shell=True)   # pyzbar's C library (optional)
    # --force-reinstall: the version string stays 0.1.0 across commits, so pip would otherwise keep a stale copy.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", "--no-cache-dir",
                    f"git+https://github.com/Rakshit-Sawarn-iitb/DiffuQR@{GIT_REF}"], check=True)
    import json as _json
    from importlib.metadata import distribution
    _url = _json.loads(distribution("diffuqr").read_text("direct_url.json") or "{}")
    print("installed diffuqr from commit", _url.get("vcs_info", {}).get("commit_id"))
    from kaggle_secrets import UserSecretsClient

    def _secret(name):
        try:
            return UserSecretsClient().get_secret(name)
        except Exception:
            return None
    HF_TOKEN = _secret("HF_TOKEN")
    SAM3_TOKEN = _secret("SAM3_HF_TOKEN") or HF_TOKEN
    WORK = Path("/kaggle/working")
else:  # local smoke run from the repo root (or notebooks/)
    root = Path.cwd() if (Path.cwd() / "diffuqr").exists() else Path.cwd().parent
    sys.path.insert(0, str(root))
    from diffuqr.config import HF_TOKEN
    SAM3_TOKEN = os.environ.get("SAM3_HF_TOKEN") or HF_TOKEN
    WORK = root / "artifacts" / "testing_local"
DATA = WORK / "data"
OUT = WORK / "testing"
OUT.mkdir(parents=True, exist_ok=True)
assert HF_TOKEN, "HF_TOKEN missing (Kaggle: Add-ons -> Secrets)"

import importlib.util
import diffuqr, torch
assert importlib.util.find_spec("diffuqr.pipeline"), "installed diffuqr has no diffuqr.pipeline: push the repo / check GIT_REF"
try:
    from pyzbar.pyzbar import decode as zbar_decode
except Exception as e:   # missing libzbar: the OpenCV column still runs
    zbar_decode = None
    print("pyzbar unavailable:", type(e).__name__, e)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("diffuqr from", Path(diffuqr.__file__).parent, "| torch", torch.__version__, "| device", DEVICE)

In [ ]:
# ---- Data: N random frames per session + ground truth (scoring only) ------------
import json, random, re
import numpy as np
from huggingface_hub import HfApi, snapshot_download

api = HfApi(token=HF_TOKEN)
assert api.repo_info(DATASET_REPO, repo_type="dataset").private, f"{DATASET_REPO} is public: stop and make it private"
repo_files = set(api.list_repo_files(DATASET_REPO, repo_type="dataset"))

rng = random.Random(SEED)
picked = {}
for sid in SESSIONS:
    frames = sorted(f for f in repo_files if f.startswith(f"frames/{sid}/") and f.endswith(".jpg"))
    assert frames, f"no frames for {sid} in {DATASET_REPO}"
    picked[sid] = sorted(rng.sample(frames, min(N_PER_SESSION, len(frames))))
    print(f"{sid}: {len(picked[sid])} of {len(frames)} saved frames")

qr_ids = sorted({sid.split("_")[0] for sid in SESSIONS})
gt_files = [f"calibration/{q}_{s}" for q in qr_ids for s in ("qrmeta.json", "modules.npy")]
missing = [f for f in gt_files if f not in repo_files]
assert not missing, f"ground truth missing on the Hub: {missing}. Run `python -m scripts.push_derived` locally."
oracle_files = [f"modspace/real_{sid}.npz" for sid in SESSIONS]
extra = [f for f in oracle_files + ["annotations/all_boxes.jsonl"] if f in repo_files]
print("no oracle file (oracle column skipped):", [f for f in oracle_files if f not in repo_files] or "none")
snapshot_download(DATASET_REPO, repo_type="dataset", token=HF_TOKEN, local_dir=str(DATA),
                  allow_patterns=[f for v in picked.values() for f in v] + gt_files + extra)

GT = {}
for q in qr_ids:
    meta = json.loads((DATA / "calibration" / f"{q}_qrmeta.json").read_text(encoding="utf-8"))
    GT[q] = {"version": meta["version"], "error": meta["error"], "payload": meta["payload"],
             "grid": np.load(DATA / "calibration" / f"{q}_modules.npy")}
    print(f"{q}: v{meta['version']} EC-{meta['error']} ({17 + 4 * meta['version']} modules)")

ORACLE = {}   # (session, frame_idx) -> reference-aligned module-space input
for sid in SESSIONS:
    p = DATA / "modspace" / f"real_{sid}.npz"
    if p.exists():
        with np.load(p, allow_pickle=True) as d:
            inputs, fidx, aligned = d["inputs"], d["frame_idx"], d["aligned"]
        ORACLE.update({(sid, int(fidx[i])): inputs[i] for i in np.flatnonzero(aligned)})
print("oracle inputs available:", len(ORACLE))
FRAME_RE = re.compile(r"_f(\d+)\.jpg$")

In [ ]:
# ---- Models: SAM3 (segmentation) + module-space diffusion checkpoint --------------
from huggingface_hub import hf_hub_download
from diffuqr.diffusion import ModelConfig, ModuleDiffusion

if MASK_SOURCE == "sam3":
    from transformers import Sam3Model, Sam3Processor
    sam3_model = Sam3Model.from_pretrained("facebook/sam3", token=SAM3_TOKEN).to(DEVICE).eval()
    sam3_proc = Sam3Processor.from_pretrained("facebook/sam3", token=SAM3_TOKEN)
    print("SAM3 loaded")
else:
    from diffuqr.annotations import pick_square_box
    BOXES = {}
    with open(DATA / "annotations" / "all_boxes.jsonl") as f:
        for line in f:
            r = json.loads(line)
            BOXES[r["file_name"]] = r

ck_path = hf_hub_download(MODEL_REPO, CKPT, token=HF_TOKEN, local_dir=str(WORK / "hub_ckpt"))
ck = torch.load(ck_path, map_location="cpu", weights_only=False)
model = ModuleDiffusion(ModelConfig(**ck["model_cfg"]))
model.load_state_dict(ck["ema"])
model = model.to(DEVICE).eval()
MODE = ck.get("mode", "diffusion")
print(f"model {CKPT}: step {ck['step']}, mode {MODE}")
del ck

In [ ]:
# ---- Stage 1: segmentation mask per frame (+ native decoders on the full frame) ----
import time, cv2
from PIL import Image
from diffuqr.pipeline import frame_gray, mask_touches_border, pick_mask, quad_from_mask


def qr_mask(image, repo_path):
    """(bool mask HxW, score) of the QR, or (None, 0.0)."""
    if MASK_SOURCE == "sam3":
        inputs = sam3_proc(images=image, text="QR Code", return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = sam3_model(**inputs)
        res = sam3_proc.post_process_instance_segmentation(
            out, threshold=SAM3_THRESHOLD, mask_threshold=SAM3_MASK_THRESHOLD,
            target_sizes=inputs.get("original_sizes").tolist())[0]
        masks = res["masks"].detach().cpu().numpy().astype(bool).reshape(-1, image.height, image.width)
        scores = res["scores"].detach().cpu().numpy().reshape(-1)
        i = pick_mask(masks, scores)
        return (None, 0.0) if i is None else (masks[i], float(scores[i]))
    rec = BOXES.get(repo_path)               # "box": the filled bounding box (no shape information)
    b = pick_square_box(rec["boxes"]) if rec else None
    if b is None:
        return None, 0.0
    m = np.zeros((image.height, image.width), bool)
    x1, y1, x2, y2 = [int(round(c)) for c in b]
    m[max(y1, 0):y2, max(x1, 0):x2] = True
    return m, 1.0


def native_status(rgb, payload):
    """Off-the-shelf decoders on the full frame: OpenCV and pyzbar, each match / wrong / none."""
    def st(txt):
        return "none" if not txt else ("match" if txt == payload else "wrong")
    txt, _, _ = cv2.QRCodeDetector().detectAndDecode(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    out = {"opencv": st(txt)}
    if zbar_decode is not None:
        res = [r.data.decode("utf-8", "replace") for r in zbar_decode(frame_gray(rgb))]
        out["pyzbar"] = "match" if payload in res else ("wrong" if res else "none")
    return out


rows, jobs = [], []          # jobs: (row index, gray frame, mask quad) for the geometry stage
t0 = time.time()
for sid in SESSIONS:
    gt = GT[sid.split("_")[0]]
    for f in picked[sid]:
        image = Image.open(DATA / f).convert("RGB")          # frames carry no EXIF rotation
        rgb = np.asarray(image)
        gray = frame_gray(rgb)
        row = {"session": sid, "frame": Path(f).name, "frame_idx": int(FRAME_RE.search(f).group(1)),
               "lap_var": float(cv2.Laplacian(gray, cv2.CV_64F).var())}
        row.update(native_status(rgb, gt["payload"]))
        ts = time.time()
        mask, score = qr_mask(image, f)
        row["t_seg_ms"] = 1000 * (time.time() - ts)
        row["mask_found"], row["mask_score"] = mask is not None, score
        quad = quad_from_mask(mask) if mask is not None else None
        row["cut_off"] = bool(mask is not None and mask_touches_border(mask))
        row["mask_quad"] = None if quad is None else quad.round(1).tolist()
        if quad is not None:
            jobs.append((len(rows), gray, quad))
        rows.append(row)
    print(f"{sid}: segmented {len(picked[sid])} frames ({time.time() - t0:.0f}s elapsed)")
print(f"masks found: {sum(r['mask_found'] for r in rows)}/{len(rows)}; quads: {len(jobs)}")

In [ ]:
# ---- Stage 2: geometry from the QR's own fixed patterns (CPU, parallel on Kaggle) ------
from multiprocessing import Pool
from diffuqr.pipeline import locate, module_input


def _locate(job):
    i, gray, quad = job
    ts = time.time()
    g = locate(gray, quad, top=GEOM_TOP)
    return i, g, 1000 * (time.time() - ts)


t0 = time.time()
if ON_KAGGLE:
    with Pool(os.cpu_count()) as pool:            # fork: workers only run numpy/OpenCV
        results = pool.map(_locate, jobs, chunksize=1)
else:
    results = [_locate(j) for j in jobs]
GEOM = {}
grays = {i: gray for i, gray, _ in jobs}
for i, g, ms in results:
    r = rows[i]
    r["t_geom_ms"] = ms
    if g is None:
        continue
    GEOM[i] = g
    v_true = GT[r["session"].split("_")[0]]["version"]
    r.update(version=g.version, version_ok=g.version == v_true, rotation=g.rotation,
             geom_score=g.score, geom_init_score=g.init_score, quad=g.quad.round(1).tolist())
print(f"geometry for {len(GEOM)}/{len(jobs)} quads in {time.time() - t0:.0f}s; "
      f"version correct {np.mean([rows[i]['version_ok'] for i in GEOM]):.0%}")

In [ ]:
# ---- Stage 3: model input -> diffusion -> RS decode (plus the learning-free reader) -------
from diffuqr.diffusion import run_eval
from diffuqr.modspace import baseline_read
from diffuqr.qrcodec import decode_soft
from diffuqr.qrgrid import modules_for_version, read_format


def reader_status(inp, version, payload):
    n = modules_for_version(version)
    bits, conf = baseline_read(inp, n, 8, 2)
    e, m, _ = read_format(bits, version)
    res, _ = decode_soft(bits, conf, version, e, m)
    return "match" if (res.ok and res.payload == payload) else ("wrong" if res.ok else "none")


INPUTS = {}   # variant -> {row index: u8 input} (kept for the figure below)
items = {"model_stretch": [], "model_raw": [], "oracle": []}
for i, r in enumerate(rows):
    gt = GT[r["session"].split("_")[0]]
    base = {"row": i, "payload": gt["payload"]}
    if i in GEOM:
        g = GEOM[i]
        grid = gt["grid"] if g.version == gt["version"] else None   # module error only when the version is right
        for name, stretch in (("model_stretch", True), ("model_raw", False)):
            inp = module_input(grays[i], g, stretch=stretch)
            INPUTS.setdefault(name, {})[i] = inp
            items[name].append({**base, "input": inp, "version": g.version, "grid": grid})
        r["reader"] = reader_status(INPUTS["model_raw"][i], g.version, gt["payload"])
    else:
        r["reader"] = "none"
    key = (r["session"], r["frame_idx"])
    if key in ORACLE:
        INPUTS.setdefault("oracle", {})[i] = ORACLE[key]
        items["oracle"].append({**base, "input": ORACLE[key], "version": gt["version"], "grid": gt["grid"]})

for name, its in items.items():
    if not its:
        continue
    t0 = time.time()
    out = run_eval(model, its, K_LIST, SAMPLE_STEPS, DEVICE, mode=MODE)
    for o in out:
        r = rows[o["row"]]
        for k in K_LIST:
            r[f"{name}_k{k}"] = o[f"verifier_k{k}"]
        r[f"{name}_err"] = o.get("module_err_k1", np.nan)
        r[f"t_{name}_ms"] = o["ms_per_item_allK"]
    print(f"{name}: {len(its)} frames in {time.time() - t0:.0f}s")

In [ ]:
# ---- Results -----------------------------------------------------------------------
import pandas as pd

res = pd.DataFrame(rows)
k = K_LIST[0]
METHODS = ["opencv"] + (["pyzbar"] if "pyzbar" in res else []) + ["reader", f"model_stretch_k{k}", f"model_raw_k{k}"]
for c in METHODS + [f"oracle_k{k}"]:
    if c not in res:
        res[c] = np.nan
    else:
        res[c] = res[c].fillna("none") if c != f"oracle_k{k}" else res[c]
native = [c for c in ("opencv", "pyzbar") if c in res]
res["native_or_model"] = np.where((res[native] == "match").any(axis=1) | (res[f"model_stretch_k{k}"] == "match"),
                                  "match", "none")
for sid in SESSIONS:   # blur tertiles within each session's sample (Laplacian variance of the full frame)
    s = res.session == sid
    rank = res.loc[s, "lap_var"].rank(method="first") - 1
    nn = s.sum()
    res.loc[s, "blur"] = np.where(rank < nn / 3, "blurriest", np.where(rank < 2 * nn / 3, "medium", "sharpest"))
res["visibility"] = np.where(~res.mask_found, "no mask", np.where(res.cut_off, "cut off", "fully in frame"))

pct = lambda s: f"{100 * (s == 'match').mean():.0f}% ({int((s == 'match').sum())}/{len(s)})"
cols = METHODS + ["native_or_model"]
L = [f"# End-to-end test: {CKPT}, masks from {MASK_SOURCE}, K={k}, {SAMPLE_STEPS} DDIM steps", ""]
for sid in SESSIONS:
    d = res[res.session == sid]
    gt = GT[sid.split("_")[0]]
    with_g = d[d.version.notna()] if "version" in d else d.iloc[:0]
    L += [f"## {sid} (v{gt['version']}, {len(d)} random frames)", "",
          f"- mask found {d.mask_found.mean():.0%}, mask touches frame edge {d.cut_off.mean():.0%}",
          f"- version detected correctly {with_g.version_ok.mean() if len(with_g) else 0:.0%} of frames with geometry; "
          f"median structure score {with_g.geom_score.median() if len(with_g) else float('nan'):.2f}",
          "", "| blur | visibility | n | " + " | ".join(cols) + " | oracle (aligned frames only) |",
          "|---|---|---|" + "---|" * (len(cols) + 1)]
    groups = [("all", "all", d)]
    for t in ("blurriest", "medium", "sharpest"):
        s = d[d.blur == t]
        groups += [(t, "all", s), (t, "fully in frame", s[s.visibility == "fully in frame"])]
    for t, v, s in groups:
        if len(s):
            o = s[f"oracle_k{k}"].dropna()
            L.append(f"| {t} | {v} | {len(s)} | " + " | ".join(pct(s[c]) for c in cols)
                     + f" | {pct(o) if len(o) else '-'} |")
    L.append("")
wrong = {c: int((res[c] == "wrong").sum()) for c in METHODS + [f"oracle_k{k}"]}
L += [f"Wrong decodes (all sessions): {wrong}", "",
      "Median time per frame: " + ", ".join(f"{c[2:-3]} {res[c].median():.0f} ms" for c in
                                            ("t_seg_ms", "t_geom_ms", "t_model_stretch_ms") if c in res), ""]
report = "\n".join(L)
print(report)
(OUT / "report.md").write_text(report, encoding="utf-8")
res.to_csv(OUT / "results.csv", index=False)      # no payload in here
print("wrote", OUT / "report.md", "and", OUT / "results.csv")

In [ ]:
# ---- Figure: frames with mask quad (red) and refined quad (green), model input, outcome ----
import matplotlib.pyplot as plt

show = []
for sid in SESSIONS:
    d = res[(res.session == sid) & res.mask_found]
    for t in ("blurriest", "medium", "sharpest"):
        show += list(d[d.blur == t].index[:2])
fig, axes = plt.subplots(len(show), 2, figsize=(9, 3.6 * len(show)), squeeze=False)
for ax, i in zip(axes, show):
    r = res.loc[i]
    im = np.asarray(Image.open(DATA / "frames" / r.session / r.frame).convert("RGB"))
    ax[0].imshow(im)
    for key, col in (("mask_quad", "r"), ("quad", "lime")):
        q = r.get(key)
        if isinstance(q, list):
            q = np.array(q + q[:1])
            ax[0].plot(q[:, 0], q[:, 1], color=col, lw=1.5)
    if isinstance(r.get("mask_quad"), list):
        q = np.array(r.mask_quad)
        pad = 0.3 * np.ptp(q[:, 0])
        ax[0].set_xlim(q[:, 0].min() - pad, q[:, 0].max() + pad)
        ax[0].set_ylim(q[:, 1].max() + pad, q[:, 1].min() - pad)
    ax[0].set_title(f"{r.session[:7]} {r.frame_idx} | {r.blur} | {r.visibility}", fontsize=9)
    inp = INPUTS.get("model_stretch", {}).get(i)
    if inp is not None:
        ax[1].imshow(inp, cmap="gray", vmin=0, vmax=255)
    ax[1].set_title(f"v{r.get('version', '-')} ok={r.get('version_ok', '-')} | model {r[f'model_stretch_k{k}']}"
                    f" | reader {r.reader} | opencv {r.opencv}", fontsize=9)
    for a in ax:
        a.axis("off")
plt.tight_layout()
plt.savefig(OUT / "examples.png", dpi=80)
plt.show()